# 04_improve

03 の勾配ブースティングに，次の 2 つをそれぞれ足して，CV で比べる．モデルの設定は 03 と同じ（既定のまま，早期終了だけ切る）．

1. 天気の前後関係：1 時間前の天気と，雨が何時間続いているか
2. casual と registered を別々に予測して足す

In [ ]:
# src/ 以下を編集したら，カーネルを再起動しなくても読み込み直す
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import PredefinedSplit, cross_val_predict, cross_val_score

from bike_sharing.cv import fold_scores, get_folds
from bike_sharing.data import load_data, save_submission
from bike_sharing.features import add_time_features, add_weather_lags
from bike_sharing.visualize import set_style

# print で DataFrame を出すときに，列が「...」で省略されないようにする
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
set_style()

train, test = load_data()
print("train:", train.shape, " test:", test.shape)

## 特徴量と目的変数

03 と同じ．

In [ ]:
train = add_time_features(train)
test = add_time_features(test)

features = ["season", "holiday", "workingday", "weather", "temp", "atemp", "humidity", "windspeed",
            "year", "month", "hour", "dow"]
X = train[features]
X_test = test[features]
y = np.log1p(train["count"])

folds = get_folds(train)
block_cv = PredefinedSplit(folds)

## 03 の CV（比べる元）

In [ ]:
gb = HistGradientBoostingRegressor(early_stopping=False)
gb_scores = -cross_val_score(gb, X, y, cv=block_cv, scoring="neg_root_mean_squared_error")
print("03:", gb_scores.round(4), " 平均", gb_scores.mean().round(4))

## 候補 1：天気の前後関係

03 までの特徴量は，その時間の天気しか見ていない．でも，雨が降り始めたばかりの時間と，何時間も降り続いている時間とでは，出かけている人の数が違うはず．そこで，1 時間前の天気（`weather_prev`）と，雨（weather が 3 以上）が何時間続いているか（`rain_hours`）を足す．

作り方は `src/bike_sharing/features.py` の `add_weather_lags` に書いた．weather は test にもあるので，train と test をつなげて 1 時間ごとに並べてから作る（test の 20 日 0 時の 1 時間前は，train の 19 日 23 時になる）．データが抜けている時間があるので，全部の時間を並べ直してから 1 時間ずらす．1 時間前が抜けている行の `weather_prev` は NaN のままにする（HistGradientBoostingRegressor は NaN をそのまま扱える）．`rain_hours` は，抜けた時間があると数え直しになる．

In [ ]:
train, test = add_weather_lags(train, test)
print("weather_prev が NaN の行:", train["weather_prev"].isna().sum(), test["weather_prev"].isna().sum())

In [ ]:
# 確認：2012-04-18 は 8 時から 15 時まで雨
example = train[(train["datetime"] >= "2012-04-18 06:00") & (train["datetime"] <= "2012-04-18 18:00")]
example[["datetime", "weather", "weather_prev", "rain_hours", "count"]]

### 03 の予測は雨の前後でどう外れていたか

足す前に，03 の out-of-fold 予測の誤差を，雨の状態で分けて見る．

In [ ]:
oof_base = cross_val_predict(gb, X, y, cv=block_cv)

# 雨の状態を 4 つに分ける
state = pd.Series("雨なし", index=train.index)
state[train["rain_hours"] == 1] = "降り始め"
state[train["rain_hours"] >= 2] = "降り続き"
state[(train["weather"] < 3) & (train["weather_prev"] >= 3)] = "止んだ直後"
state.value_counts()

In [ ]:
# RMSLE と，予測 - 実際 の平均（log の空間．プラスなら多めに予測している）
err = pd.DataFrame({"state": state, "03": (oof_base - y) ** 2})
bias = pd.DataFrame({"state": state, "03": oof_base - y})

print("RMSLE")
print(np.sqrt(err.groupby("state").mean()).round(3))
print()
print("予測 - 実際 の平均")
print(bias.groupby("state").mean().round(3))

### 候補 1 の CV

In [ ]:
features_1 = features + ["weather_prev", "rain_hours"]

scores_1 = -cross_val_score(gb, train[features_1], y, cv=block_cv, scoring="neg_root_mean_squared_error")
print("候補 1:", scores_1.round(4), " 平均", scores_1.mean().round(4))

In [ ]:
# 雨の状態ごとに，03 と比べる
oof_1 = cross_val_predict(gb, train[features_1], y, cv=block_cv)
err["候補 1"] = (oof_1 - y) ** 2
bias["候補 1"] = oof_1 - y

print("RMSLE")
print(np.sqrt(err.groupby("state").mean()).round(3))
print()
print("予測 - 実際 の平均")
print(bias.groupby("state").mean().round(3))

In [ ]:
b = bias.groupby("state").mean().loc[["雨なし", "降り始め", "降り続き", "止んだ直後"]]
x = np.arange(len(b))

plt.figure(figsize=(7, 4))
plt.bar(x - 0.2, b["03"], width=0.4, label="03")
plt.bar(x + 0.2, b["候補 1"], width=0.4, label="候補 1")
plt.xticks(x, b.index)
plt.ylabel("予測 - 実際（log の平均）")
plt.title("雨の状態ごとの予測のずれ")
plt.legend()
plt.show()

## 候補 2：casual と registered に分ける

01_eda の 3 節で見たとおり，registered は勤務日の朝と夕方に山があり，casual は土日の昼に多い．形の違う 2 つの合計を 1 つのモデルで学習するより，別々に学習して足したほうが当たるかもしれない．train では count = casual + registered なので，それぞれの log(1 + x) を学習し，台数に戻して足す．

特徴量は 03 と同じにする．

In [ ]:
y_casual = np.log1p(train["casual"])
y_registered = np.log1p(train["registered"])

oof_casual = cross_val_predict(gb, X, y_casual, cv=block_cv)
oof_registered = cross_val_predict(gb, X, y_registered, cv=block_cv)

# 台数に戻して足し，log(1 + count) にして比べる
pred_count = np.expm1(oof_casual) + np.expm1(oof_registered)
print("0 未満になった行:", (pred_count < 0).sum())
oof_2 = np.log1p(pred_count)

# fold ごとの RMSLE（cross_val_score と同じ計算）
scores_2 = fold_scores(y, oof_2, folds)
print("候補 2:", scores_2.round(4), " 平均", scores_2.mean().round(4))

In [ ]:
# casual と registered，それぞれの当たり具合
print("casual の RMSLE:", round(root_mean_squared_error(y_casual, oof_casual), 4))
print("registered の RMSLE:", round(root_mean_squared_error(y_registered, oof_registered), 4))
print("count に占める casual の割合:", round(train["casual"].sum() / train["count"].sum(), 3))

## 両方

候補 1 の特徴量で，casual と registered を別々に予測する．

In [ ]:
oof_casual_both = cross_val_predict(gb, train[features_1], y_casual, cv=block_cv)
oof_registered_both = cross_val_predict(gb, train[features_1], y_registered, cv=block_cv)

pred_count = np.expm1(oof_casual_both) + np.expm1(oof_registered_both)
print("0 未満になった行:", (pred_count < 0).sum())
oof_both = np.log1p(pred_count)

scores_both = fold_scores(y, oof_both, folds)
print("両方:", scores_both.round(4), " 平均", scores_both.mean().round(4))

## CV のまとめ

In [ ]:
result = pd.DataFrame({
    "03": gb_scores,
    "候補 1": scores_1,
    "候補 2": scores_2,
    "両方": scores_both,
}, index=["1〜5 日", "6〜10 日", "11〜14 日", "15〜19 日"])
result.loc["平均"] = result.mean()
result.round(4)

In [ ]:
# 03 との差（マイナスなら良くなった）
diff = pd.DataFrame(index=result.index)
for name in ["候補 1", "候補 2", "両方"]:
    diff[name] = result[name] - result["03"]
diff.round(4)

## 提出用のファイル

作っておくだけで，提出は Kaggle の画面からする．候補 1，候補 2，両方の 3 つを書き出す．

In [ ]:
# 全部の train で学習し直して test を予測する
# 候補 1
gb.fit(train[features_1], y)
pred_1 = np.expm1(gb.predict(test[features_1]))
print("0 未満になった行:", (pred_1 < 0).sum())

save_submission(pd.DataFrame({"datetime": test["datetime"], "count": pred_1}), "04_improve_weather")

In [ ]:
# 候補 2
gb_casual = HistGradientBoostingRegressor(early_stopping=False)
gb_registered = HistGradientBoostingRegressor(early_stopping=False)
gb_casual.fit(X, y_casual)
gb_registered.fit(X, y_registered)
pred_2 = np.expm1(gb_casual.predict(X_test)) + np.expm1(gb_registered.predict(X_test))
print("0 未満になった行:", (pred_2 < 0).sum())

save_submission(pd.DataFrame({"datetime": test["datetime"], "count": pred_2}), "04_improve_split")

In [ ]:
# 両方
gb_casual.fit(train[features_1], y_casual)
gb_registered.fit(train[features_1], y_registered)
pred_both = np.expm1(gb_casual.predict(test[features_1])) + np.expm1(gb_registered.predict(test[features_1]))
print("0 未満になった行:", (pred_both < 0).sum())

save_submission(pd.DataFrame({"datetime": test["datetime"], "count": pred_both}), "04_improve_both")

## まとめ

- 候補 1（1 時間前の天気と，雨が何時間続いているか）で，CV は 0.3172 から 0.3060 になり，4 つの fold すべてで良くなった．
- 03 は，雨の降り始めを少なめに，降り続いている時間と止んだ直後を多めに予測していた（予測 - 実際 の平均が −0.16，0.20，0.20）．降り始めはまだ人が出ていて，降り続くと減り，止んでもすぐには戻らない，という動きだと思う．候補 1 を足すと，このずれは 0.03，0.08，0.02 まで小さくなった．
- 候補 2（casual と registered に分ける）は −0.0018 で，6〜10 日と 15〜19 日の fold では逆に悪くなった．casual は count の 19% しかなく，予測も外れやすい（RMSLE 0.52．registered は 0.32）．
- 両方入れると 0.3046 で，候補 1 だけとの差は 0.0014．